## Experiment design

### Experiment Design Idea

我们要检测模型是否理解语法，当有语法的对应数据集，需要做两种测试。


1. 数据进行组合增强， 然后加入一定的干扰，比如说某些单词随机交换位置。  （一个表，统计数据，讲述来源）
2. 语法 加 文本：训练LLM 输入文本，然后让模型指出语法，让模型进行选择，单个语法，做分类。 （一个表，用来解释单分类 （句子 -》 语法））
3. 对语法进行多个语法的组合，进行多标签的识别，看一下模型是否认识或者识别了语法的功能，输入是多个句子的组合（包含不同语法）， 输出是做分类。 （一个表，用来描述多分类，句子 -》 语法）
4. 反着来，给他输入语法，然后给他混杂一定的句子，看他们是否能够分辨出那些句子是使用了这个语法，那些没有。 （一个表 （语法 -》句子））
5. 用上我们的训练好的模型，问模型平行语料的训练是否可以让模型实现语法的学习和训练。 （一个表，用来解释这些东西 （平行语料是否可以让模型学习到语法？））
6. 强化学习，让模型翻译好的初步测试，还是让模型进行分类。 （强化学习，让模型进行分类，让模型分析，并对句子所使用的语法进行列举 (训练一个强化学习的模型让他来对句子，来分析使用了那些语法)）

We want to test whether the model understands grammar. When grammar-related datasets are available, we should conduct several types of tests:

1. Perform data augmentation through combinations, then introduce some noise, such as randomly exchanging positions of certain words. (Use a table to record statistics and explain the source of the data)

2. Grammar + text: Train the LLM with input text, then have the model identify the grammar and make a choice. Each case involves a single grammar category for classification. (Use a table to explain single-class classification)

3. Combine multiple grammar points and perform multi-label recognition to see whether the model can recognize or identify the functions of different grammar structures. Multiple grammar items are classified together. (Use a table to describe multi-class classification)

4. Reverse the approach: provide the grammar first, then mix in several sentences, and test whether the model can distinguish which sentences use the grammar and which do not. (Use a table to explain advanced tasks)

5. Use our trained model to test whether training with parallel corpora enables the model to perform grammar learning and training. (Use a table to explain this method)

6. Reinforcement learning: conduct preliminary testing with translation tasks, but also have the model continue with classification. (Reinforcement learning for grammar classification: let the model analyze and list the grammatical structures used in the sentences)




## Data Processing

In [3]:
import pandas as pd

df = pd.read_json("data/extraction_pdf/extraction_table_texts/extraction_20250925_162528.jsonl", lines=True)


In [4]:
count = 0
count_examples = 0
for i, row in df.iterrows():
    row_output_dict = row["output_dict"]
    if 'grammars' not in row_output_dict or row_output_dict['grammars'] is None:
        print("No grammars")
    else:
        for grammar in row_output_dict['grammars']:
            print (grammar["description"])
            count += 1
            for example in grammar["examples"]:
                count_examples += 1

print (f"Total grammars: {count}")
print (f"Total examples: {count_examples}")

In Luxembourgish compounds, the original and historically common pattern places primary stress on the second constituent of the compound (i.e., the right-hand element). Examples from the literature show lexical items and adjectives with final-element stress, but ongoing language contact with German has caused a gradual tendency to shift stress toward the first constituent. This rule applies to both nominal and adjectival compounds and can be observed in synchronic variation across speakers and registers.
Luxembourgish exhibits a distinctive intonational contour not found in German or French: the pitch rises toward the nuclear syllable of a phrase and instead of maintaining a high plateau or falling at the end, it drops to a mid-high level and remains roughly constant until the phrase boundary. This contour is frequent and typologically noteworthy; recent intonation research (starting in the 2010s) has documented its phonetic realization and its pragmatic functions in connected speech.


### Data Augmentation For Experiements



In [21]:
import pandas as pd

df = pd.read_json("data/extraction_pdf/extraction_table_texts/extraction_20250925_162528.jsonl", lines=True)

In [22]:
df_grammars = pd.DataFrame()
df_samples = pd.DataFrame()
count_grammars = 0
count_examples = 0
for i, row in df.iterrows():
    row_output_dict = row["output_dict"]
    if 'grammars' not in row_output_dict or row_output_dict['grammars'] is None:
        print("No grammars")
    else:
        grammar_dict = {}
        for grammar in row_output_dict['grammars']:
            grammar_dict["grammar_points_descriptions"] = grammar["description"]
            grammar_dict["idx"] =  count_grammars ; count_grammars += 1
            list_examples = []
            sample_dict = {}
            for d in grammar["examples"]:
                num = list(d.keys())[0].split("_")[1]  # 取编号 1/2/3
                lux = d[f"example_{num}_luxembourg"]
                eng = d[f"example_{num}_english"]
                sample_dict["luxembourg"] = lux
                sample_dict["english"] = eng
                sample_dict["grammar_points_descriptions"] = grammar["description"]
                sample_dict["idx_samples"] = count_examples; count_examples += 1
                sample_dict["idx_grammar"] = grammar_dict["idx"]
                df_samples = pd.concat([df_samples, pd.DataFrame([sample_dict])], ignore_index=True)
                list_examples.append(sample_dict)
                
            grammar_dict["examples"] = list_examples
        df_grammars = pd.concat([df_grammars, pd.DataFrame([grammar_dict])], ignore_index=True)


In [23]:
df_grammars

,grammar_points_descriptions,idx,examples
0,Luxembourgish exhibits a distinctive intonatio...,1,[{'luxembourg': 'Zënter d'Fuerschung un Intona...
1,The morphosyntactic system of Luxembourgish ov...,3,[{'luxembourg': 'An erzielte Memoiren iwwer Ka...
2,"The passage notes that, due to space restricti...",7,[{'luxembourg': 'Fir komplett Aspekter wéi d'F...
3,Loans from French are frequently integrated in...,11,[{'luxembourg': 'Wéinst staarker franséischer ...
4,"In Luxembourgish, personal pronouns referring ...",13,[{'luxembourg': 'Wann de Moler erzielt datt sä...
...,...,...,...
261,Apparent-time analysis compares speech of diff...,654,"[{'luxembourg': 'Fuerscher, déi Apparent-Time-..."
262,This rule highlights sociolinguistic patterns ...,658,[{'luxembourg': 'Feldaarbecht mat Alterskohort...
263,Luxembourgish forms the comparative by placing...,662,[{'luxembourg': 'Fir vill erwuesse Léierender ...
264,"Despite orthographic standardization, the impl...",666,[{'luxembourg': 'Obwuel digital Ressourcen ver...


In [25]:
df_samples.to_json("data/extraction_pdf/datasets/df_samples.jsonl", lines=True, orient="records")
df_grammars.to_json("data/extraction_pdf/datasets/df_grammars.jsonl", lines=True, orient="records")

## Data Augementation

In [ ]:
import pandas as pd
from itertools import combinations

grouped = df_samples.groupby("idx_grammar")

augmented_data = []

for gram_idx, group in grouped:
    sentences_lux = group["luxembourg"].tolist()
    sentences_en = group["english"].tolist()
    
    for i, j, k in combinations(range(len(sentences_lux)), 3):
        lux_combo = sentences_lux[i] + " " + sentences_lux[j] + " " + sentences_lux[k]
        eng_combo = sentences_en[i] + " " + sentences_en[j] + " " + sentences_en[k]
        
        augmented_data.append({
            "luxembourg": lux_combo,
            "english": eng_combo,
            "grammar_points_descriptions": group["grammar_points_descriptions"].iloc[0],
            "idx_samples": f"{gram_idx}_{i}_{j}_{k}",
            "idx_grammar": gram_idx
        })

    for i, j in combinations(range(len(sentences_lux)), 2):
        lux_combo = sentences_lux[i] + " " + sentences_lux[j]
        eng_combo = sentences_en[i] + " " + sentences_en[j]
        
        augmented_data.append({
            "luxembourg": lux_combo,
            "english": eng_combo,
            "grammar_points_descriptions": group["grammar_points_descriptions"].iloc[0],
            "idx_samples": f"{gram_idx}_{i}_{j}",
            "idx_grammar": gram_idx
        })

df_augmented = pd.DataFrame(augmented_data)

df_final = pd.concat([df_samples, df_augmented], ignore_index=True)
print(df_final.shape)


(4858, 5)


In [31]:
df_final.to_json("data/extraction_pdf/datasets/df_samples_augmented_combo_3_2.jsonl", lines=True, orient="records")

## Models training

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_parts, val_parts, test_parts = [], [], []

for gram_idx, group in df_samples.groupby("idx_grammar"):
    train, temp = train_test_split(group, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    
    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

train_df = pd.concat(train_parts, ignore_index=True)
val_df   = pd.concat(val_parts, ignore_index=True)
test_df  = pd.concat(test_parts, ignore_index=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))
print("Grammars situations：")
print("Train:", train_df['idx_grammar'].nunique())
print("Val:", val_df['idx_grammar'].nunique())
print("Test:", test_df['idx_grammar'].nunique())


ValueError: With n_samples=1, test_size=0.5 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.